# 遷移學習實戰 (Transfer Learning in Practice)
:label:`sec_transfer_learning`

在實際應用中，我們很少會從頭開始訓練整個卷積神經網路（隨機初始化），因為：

1. **數據不足**：獲取足夠的標註數據非常困難和昂貴
2. **計算資源**：從零訓練大型網路需要大量的時間和GPU資源
3. **更好的效果**：使用預訓練模型通常能獲得更好的性能

**遷移學習**（Transfer Learning）是深度學習中最實用的技術之一。本節將全面介紹遷移學習的理論與實踐。

## 本節內容

1. 遷移學習的理論基礎
2. 使用預訓練模型的策略
3. Fine-tuning 技巧
4. 實戰案例：貓狗分類
5. 使用 timm 庫快速實驗
6. 最佳實踐與常見問題

## 1. 遷移學習理論基礎

### 1.1 什麼是遷移學習？

遷移學習是將在**源任務**（如 ImageNet 分類）上學習到的知識應用到**目標任務**（如貓狗分類）的技術。

![遷移學習概念圖](../img/transfer_learning_concept.svg)

### 1.2 為什麼遷移學習有效？

CNN 學習到的特徵具有**層次性**：

- **淺層**：邊緣、紋理、顏色等通用低級特徵
- **中層**：形狀、圖案等中級特徵  
- **深層**：特定於任務的高級特徵（如貓的臉、狗的耳朵）

```
層次     特徵類型              遷移性
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Layer 1  邊緣、顏色           ✓✓✓ 高度通用
Layer 2  紋理、簡單形狀       ✓✓✓ 高度通用
Layer 3  複雜形狀、圖案       ✓✓  較通用
Layer 4  物體部件             ✓   任務相關
Layer 5  特定物體             ✗   任務特定
```

### 1.3 遷移學習的策略

根據**數據量**和**任務相似度**選擇策略：

| 數據量 | 任務相似度 | 推薦策略 | 說明 |
|--------|-----------|----------|------|
| 小 | 高 | 僅訓練分類器 | 凍結所有卷積層，只訓練全連接層 |
| 小 | 低 | 訓練淺層+分類器 | 凍結深層，訓練淺層和分類器 |
| 大 | 高 | Fine-tune 全網路 | 較小學習率微調所有層 |
| 大 | 低 | Fine-tune 深層 | 凍結淺層，微調深層和分類器 |

### 1.4 核心概念

- **特徵提取**（Feature Extraction）：凍結預訓練模型，僅訓練新添加的層
- **微調**（Fine-tuning）：以較小學習率訓練整個網路或部分層
- **漸進式解凍**（Progressive Unfreezing）：逐步解凍並訓練更多層

## 2. 環境準備

In [ ]:
# 安裝必要的套件（如果尚未安裝）
# !pip install torch torchvision timm matplotlib pillow scikit-learn tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision
from torchvision import transforms, models
import timm

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, confusion_matrix
import seaborn as sns

# 設定設備
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用設備: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# 設定隨機種子
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    
set_seed(42)

## 3. 策略一：特徵提取（Feature Extraction）

### 3.1 載入預訓練模型

我們使用在 ImageNet 上預訓練的 ResNet-50 作為特徵提取器。

In [ ]:
# 方法1: 使用 torchvision
def create_feature_extractor_torchvision(num_classes=2):
    """
    創建特徵提取器（凍結預訓練層）
    
    Args:
        num_classes: 目標任務的類別數
    """
    # 載入預訓練的 ResNet-50
    model = models.resnet50(pretrained=True)
    
    # 凍結所有參數
    for param in model.parameters():
        param.requires_grad = False
    
    # 替換最後的全連接層
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, num_classes)
    
    return model

# 創建模型
model_feature_extractor = create_feature_extractor_torchvision(num_classes=2)
model_feature_extractor = model_feature_extractor.to(device)

# 檢查可訓練參數
trainable_params = sum(p.numel() for p in model_feature_extractor.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model_feature_extractor.parameters())

print(f"可訓練參數: {trainable_params:,}")
print(f"總參數數: {total_params:,}")
print(f"可訓練比例: {trainable_params/total_params*100:.2f}%")

### 3.2 使用 timm 庫（推薦）

[timm](https://github.com/rwightman/pytorch-image-models) (PyTorch Image Models) 是一個優秀的預訓練模型庫，提供了:

- 600+ 預訓練模型
- 統一的 API
- 最新的模型架構
- 更好的性能

In [ ]:
# 查看可用的模型
available_models = timm.list_models('*resnet*', pretrained=True)
print(f"\n可用的 ResNet 模型: {len(available_models)} 個")
print("\n部分模型:")
for model_name in available_models[:10]:
    print(f"  - {model_name}")

# 使用 timm 創建模型
def create_feature_extractor_timm(model_name='resnet50', num_classes=2, freeze=True):
    """
    使用 timm 創建特徵提取器
    
    Args:
        model_name: 模型名稱
        num_classes: 目標類別數
        freeze: 是否凍結預訓練層
    """
    # 創建模型（自動下載預訓練權重）
    model = timm.create_model(
        model_name,
        pretrained=True,
        num_classes=num_classes
    )
    
    if freeze:
        # 凍結除分類器外的所有層
        for name, param in model.named_parameters():
            if 'fc' not in name and 'classifier' not in name and 'head' not in name:
                param.requires_grad = False
    
    return model

# 創建模型
model_timm = create_feature_extractor_timm('resnet50', num_classes=2)
model_timm = model_timm.to(device)

print("\n模型創建成功！")

### 3.3 數據準備

對於遷移學習，數據預處理非常重要：

1. **使用與預訓練相同的標準化參數**
2. **適當的數據增強**
3. **合適的圖像大小**

In [ ]:
# ImageNet 的標準化參數
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# 訓練集轉換（包含數據增強）
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224),  # 隨機裁剪和縮放
    transforms.RandomHorizontalFlip(),   # 隨機水平翻轉
    transforms.ColorJitter(               # 顏色抖動
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.1
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# 驗證/測試集轉換（不需要數據增強）
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

print("數據轉換準備完成！")

### 3.4 使用 CIFAR-10 作為示例數據集

為了演示，我們使用 CIFAR-10 數據集的貓和狗類別。

In [ ]:
# 創建簡化的貓狗數據集（使用 CIFAR-10）
class CatDogDataset(Dataset):
    """從 CIFAR-10 提取貓和狗的數據集"""
    
    def __init__(self, cifar_dataset, transform=None):
        # CIFAR-10 中貓的標籤是 3，狗的標籤是 5
        cat_dog_indices = [
            i for i, (_, label) in enumerate(cifar_dataset)
            if label in [3, 5]
        ]
        
        self.data = [cifar_dataset[i][0] for i in cat_dog_indices]
        self.labels = [
            0 if cifar_dataset[i][1] == 3 else 1  # 貓: 0, 狗: 1
            for i in cat_dog_indices
        ]
        self.transform = transform
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        image = self.data[idx]
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

# 載入 CIFAR-10
cifar_train = torchvision.datasets.CIFAR10(
    root='../data',
    train=True,
    download=True
)

cifar_test = torchvision.datasets.CIFAR10(
    root='../data',
    train=False,
    download=True
)

# 創建貓狗數據集
train_dataset = CatDogDataset(cifar_train, transform=train_transform)
test_dataset = CatDogDataset(cifar_test, transform=val_transform)

# 創建數據加載器
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

print(f"訓練集大小: {len(train_dataset)}")
print(f"測試集大小: {len(test_dataset)}")
print(f"類別: 貓 (0), 狗 (1)")

### 3.5 訓練特徵提取器

In [ ]:
def train_model(model, train_loader, test_loader, num_epochs=10, lr=0.001):
    """
    訓練模型
    
    Args:
        model: 要訓練的模型
        train_loader: 訓練數據加載器
        test_loader: 測試數據加載器
        num_epochs: 訓練輪數
        lr: 學習率
    """
    # 損失函數和優化器
    criterion = nn.CrossEntropyLoss()
    
    # 只優化需要梯度的參數
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr
    )
    
    # 學習率調度器
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', patience=2, factor=0.5, verbose=True
    )
    
    # 記錄訓練歷史
    history = {
        'train_loss': [],
        'train_acc': [],
        'test_acc': []
    }
    
    # 訓練循環
    for epoch in range(num_epochs):
        # 訓練階段
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            
            # 前向傳播
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # 反向傳播
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            # 統計
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()
            
            # 更新進度條
            pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100.*train_correct/train_total:.2f}%'
            })
        
        # 計算訓練指標
        epoch_loss = train_loss / len(train_loader)
        epoch_acc = 100. * train_correct / train_total
        
        # 評估階段
        model.eval()
        test_correct = 0
        test_total = 0
        
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = outputs.max(1)
                test_total += labels.size(0)
                test_correct += predicted.eq(labels).sum().item()
        
        test_acc = 100. * test_correct / test_total
        
        # 記錄歷史
        history['train_loss'].append(epoch_loss)
        history['train_acc'].append(epoch_acc)
        history['test_acc'].append(test_acc)
        
        # 更新學習率
        scheduler.step(test_acc)
        
        # 打印結果
        print(f'\nEpoch {epoch+1}/{num_epochs}:')
        print(f'  Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.2f}%')
        print(f'  Test Acc: {test_acc:.2f}%\n')
    
    return history

# 訓練特徵提取器
print("開始訓練特徵提取器...\n")
history_fe = train_model(
    model_timm,
    train_loader,
    test_loader,
    num_epochs=10,
    lr=0.001
)

## 4. 策略二：Fine-tuning（微調）

Fine-tuning 是在特徵提取的基礎上，解凍部分或全部預訓練層，以較小的學習率進行訓練。

### 4.1 創建 Fine-tuning 模型

In [ ]:
def create_finetuning_model(model_name='resnet50', num_classes=2, 
                           freeze_layers=0):
    """
    創建 Fine-tuning 模型
    
    Args:
        model_name: 模型名稱
        num_classes: 類別數
        freeze_layers: 凍結的層數（從底層開始）
                      0 = 全部可訓練
                      -1 = 只訓練分類器
    """
    model = timm.create_model(
        model_name,
        pretrained=True,
        num_classes=num_classes
    )
    
    if freeze_layers == -1:
        # 凍結所有層，只訓練分類器
        for name, param in model.named_parameters():
            if 'fc' not in name and 'classifier' not in name and 'head' not in name:
                param.requires_grad = False
    elif freeze_layers > 0:
        # 凍結前 N 層
        layers = list(model.named_parameters())
        for i, (name, param) in enumerate(layers[:freeze_layers]):
            param.requires_grad = False
    
    return model

# 創建 Fine-tuning 模型（全部可訓練）
model_finetune = create_finetuning_model(
    'resnet50',
    num_classes=2,
    freeze_layers=0  # 全部可訓練
)
model_finetune = model_finetune.to(device)

# 檢查參數
trainable = sum(p.numel() for p in model_finetune.parameters() if p.requires_grad)
total = sum(p.numel() for p in model_finetune.parameters())
print(f"可訓練參數: {trainable:,} ({trainable/total*100:.1f}%)")

### 4.2 Fine-tuning 的關鍵技巧

1. **使用較小的學習率**：通常是從頭訓練的 1/10 或 1/100
2. **差異化學習率**：不同層使用不同的學習率
3. **漸進式解凍**：先訓練分類器，再逐步解凍更多層

In [ ]:
def get_optimizer_with_differential_lr(model, base_lr=1e-4, 
                                      classifier_lr_multiplier=10):
    """
    創建差異化學習率的優化器
    
    Args:
        model: 模型
        base_lr: 基礎學習率（用於預訓練層）
        classifier_lr_multiplier: 分類器學習率倍數
    """
    # 分離參數
    classifier_params = []
    backbone_params = []
    
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        
        # 判斷是否為分類器參數
        if any(x in name for x in ['fc', 'classifier', 'head']):
            classifier_params.append(param)
        else:
            backbone_params.append(param)
    
    # 創建優化器
    optimizer = optim.Adam([
        {'params': backbone_params, 'lr': base_lr},
        {'params': classifier_params, 'lr': base_lr * classifier_lr_multiplier}
    ])
    
    print(f"Backbone 學習率: {base_lr}")
    print(f"Classifier 學習率: {base_lr * classifier_lr_multiplier}")
    
    return optimizer

# 創建優化器
optimizer_finetune = get_optimizer_with_differential_lr(
    model_finetune,
    base_lr=1e-4,
    classifier_lr_multiplier=10
)

### 4.3 訓練 Fine-tuning 模型

In [ ]:
# 使用自定義優化器訓練
def train_with_custom_optimizer(model, train_loader, test_loader, 
                                optimizer, num_epochs=10):
    """使用自定義優化器訓練模型"""
    criterion = nn.CrossEntropyLoss()
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    
    history = {'train_loss': [], 'train_acc': [], 'test_acc': []}
    
    for epoch in range(num_epochs):
        # 訓練
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for images, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}'):
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()
        
        # 評估
        model.eval()
        test_correct = 0
        test_total = 0
        
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = outputs.max(1)
                test_total += labels.size(0)
                test_correct += predicted.eq(labels).sum().item()
        
        # 記錄
        epoch_loss = train_loss / len(train_loader)
        epoch_acc = 100. * train_correct / train_total
        test_acc = 100. * test_correct / test_total
        
        history['train_loss'].append(epoch_loss)
        history['train_acc'].append(epoch_acc)
        history['test_acc'].append(test_acc)
        
        scheduler.step()
        
        print(f'Epoch {epoch+1}: Train Acc={epoch_acc:.2f}%, Test Acc={test_acc:.2f}%')
    
    return history

# 訓練 Fine-tuning 模型
print("開始 Fine-tuning...\n")
history_ft = train_with_custom_optimizer(
    model_finetune,
    train_loader,
    test_loader,
    optimizer_finetune,
    num_epochs=10
)

## 5. 性能對比與可視化

In [ ]:
def plot_training_history(histories, labels):
    """
    繪製訓練歷史對比圖
    
    Args:
        histories: 訓練歷史列表
        labels: 標籤列表
    """
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # 損失曲線
    for history, label in zip(histories, labels):
        axes[0].plot(history['train_loss'], marker='o', label=label)
    axes[0].set_title('Training Loss', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # 準確率曲線
    for history, label in zip(histories, labels):
        axes[1].plot(history['test_acc'], marker='s', label=f'{label} (Test)')
        axes[1].plot(history['train_acc'], marker='o', 
                    linestyle='--', alpha=0.6, label=f'{label} (Train)')
    axes[1].set_title('Accuracy', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# 繪製對比圖
plot_training_history(
    [history_fe, history_ft],
    ['Feature Extraction', 'Fine-tuning']
)

## 6. 實用技巧與最佳實踐

### 6.1 如何選擇預訓練模型？

| 場景 | 推薦模型 | 理由 |
|------|---------|------|
| 高精度需求 | EfficientNet-B7, ResNet-152 | 大模型，高準確率 |
| 速度優先 | MobileNetV3, EfficientNet-B0 | 輕量級，快速推理 |
| 平衡性能 | ResNet-50, EfficientNet-B3 | 準確率和速度平衡 |
| 醫學影像 | DenseNet-121 | 特徵重用，適合細粒度 |
| 移動端部署 | MobileNetV2/V3 | 專為移動端設計 |

### 6.2 學習率設置指南

```python
# 經驗法則
from_scratch_lr = 0.1       # 從頭訓練
feature_extraction_lr = 0.001  # 特徵提取（只訓練分類器）
finetuning_lr = 0.0001      # Fine-tuning（訓練全網路）

# 差異化學習率（推薦）
backbone_lr = 1e-5    # 預訓練層
classifier_lr = 1e-3  # 新添加的層
```

### 6.3 數據增強策略

```python
# 基礎增強（推薦用於遷移學習）
basic_augmentation = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# 強增強（數據非常少時）
strong_augmentation = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.4, 0.4, 0.4, 0.2),
    transforms.RandomRotation(15),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])
```

### 6.4 常見問題解決

#### 問題1: 訓練不收斂
- 檢查學習率（可能太大）
- 確保使用正確的標準化參數
- 嘗試降低批次大小

#### 問題2: 過擬合嚴重
- 增加數據增強
- 使用 Dropout
- 減少 Fine-tuning 的層數
- 使用正則化

#### 問題3: 記憶體不足
- 減小批次大小
- 使用混合精度訓練
- 選擇更小的模型
- 使用梯度累積

## 7. 進階主題：使用 timm 的高級功能

### 7.1 快速模型比較

In [ ]:
def compare_models(model_names, num_classes=2, input_size=(1, 3, 224, 224)):
    """
    比較不同模型的參數量和速度
    
    Args:
        model_names: 模型名稱列表
        num_classes: 類別數
        input_size: 輸入尺寸
    """
    results = []
    
    for name in model_names:
        try:
            model = timm.create_model(name, pretrained=False, num_classes=num_classes)
            model.eval()
            
            # 計算參數量
            params = sum(p.numel() for p in model.parameters())
            
            # 測試推理速度
            x = torch.randn(input_size)
            
            import time
            start = time.time()
            with torch.no_grad():
                for _ in range(100):
                    _ = model(x)
            elapsed = (time.time() - start) / 100
            
            results.append({
                'Model': name,
                'Parameters': f'{params/1e6:.2f}M',
                'Inference Time': f'{elapsed*1000:.2f}ms'
            })
        except Exception as e:
            print(f"Error with {name}: {e}")
    
    import pandas as pd
    df = pd.DataFrame(results)
    return df

# 比較常用模型
models_to_compare = [
    'resnet18',
    'resnet50',
    'efficientnet_b0',
    'mobilenetv3_large_100'
]

comparison = compare_models(models_to_compare)
print("\n模型對比：")
print(comparison.to_string(index=False))

### 7.2 自動選擇最佳模型

In [ ]:
def auto_select_model(requirement='balanced'):
    """
    根據需求自動選擇模型
    
    Args:
        requirement: 'speed', 'accuracy', 'balanced'
    """
    recommendations = {
        'speed': {
            'model': 'mobilenetv3_small_100',
            'reason': '最輕量級，適合移動端和實時應用',
            'lr': 1e-3,
            'batch_size': 64
        },
        'accuracy': {
            'model': 'efficientnet_b7',
            'reason': '高準確率，適合離線處理',
            'lr': 1e-4,
            'batch_size': 16
        },
        'balanced': {
            'model': 'resnet50',
            'reason': '經典架構，性能和速度平衡',
            'lr': 1e-3,
            'batch_size': 32
        }
    }
    
    config = recommendations[requirement]
    
    print(f"\n推薦配置（{requirement}）：")
    print(f"  模型: {config['model']}")
    print(f"  理由: {config['reason']}")
    print(f"  建議學習率: {config['lr']}")
    print(f"  建議批次大小: {config['batch_size']}")
    
    return config

# 示例
config = auto_select_model('balanced')

## 8. 小結

### 關鍵要點

1. **遷移學習是實用技術**
   - 幾乎所有實際應用都使用預訓練模型
   - 能大幅減少訓練時間和數據需求
   - 通常能獲得更好的性能

2. **選擇合適的策略**
   - 數據少 → 特徵提取
   - 數據多 → Fine-tuning
   - 任務相似 → 凍結更多層
   - 任務不同 → 訓練更多層

3. **學習率很關鍵**
   - Fine-tuning 使用較小學習率
   - 不同層可以使用不同學習率
   - 新層學習率 > 預訓練層學習率

4. **數據預處理要正確**
   - 使用與預訓練相同的標準化
   - 適當的數據增強
   - 合適的輸入尺寸

### 下一步學習

- [現代架構](9_modern_architectures.ipynb)：了解 EfficientNet、MobileNet 等現代架構
- [模型可視化](10_model_visualization.ipynb)：學習如何解釋模型決策
- [實用技巧](11_practical_tips.ipynb)：掌握更多優化和部署技巧

### 練習題

1. 嘗試使用不同的預訓練模型（如 EfficientNet, MobileNet）比較性能
2. 實現漸進式解凍策略（先訓練分類器，再逐步解凍更多層）
3. 在你自己的數據集上應用遷移學習
4. 比較 Feature Extraction vs Fine-tuning 在不同數據量下的表現

## 參考資源

- [timm 文檔](https://timm.fast.ai/)
- [PyTorch Transfer Learning Tutorial](https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)
- [CS231n: Transfer Learning](http://cs231n.github.io/transfer-learning/)
- [How transferable are features in deep neural networks?](https://arxiv.org/abs/1411.1792)